<a href="https://colab.research.google.com/github/bhardwaj-amit79/Colab/blob/main/%5BUpgrad_Mod_2%5D_Lec_2_Prompt_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 Lecture 2
---
# Prompt Optimization

A basic walkthrough of:
- Section 0 - Setup
- Section 1 - Load model, define helper functions
- Section 2 - Baseline - Naive prompting
- Section 3 - Role Playing
- Section 4 - Prompt Chaining
- Section 5 - Automatic Prompt Engineer
- Section 6 - Gradient-based textural optimization
- Section 7 [Try Yourself] - DSPy
- Section 8 - Conclusion

## Section 0: Setup

In [ ]:
# For local environments, install required libraries
# !pip install transformers accelerate

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Section 1: Load model, define helper functions

As discussed in a previous session, we use the pipeline library of HuggingFace to interact with the LLM.

In [ ]:
from transformers import pipeline

model_pipeline = pipeline(
    "text-generation",
    model="unsloth/Llama-3.2-3B-Instruct",
    max_new_tokens=500,
    temperature=0.1,
    top_k=10,
    top_p=0.85,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Helper functions

In [ ]:
def make_message(system_content, user_content):
    """Wrap system + user text into the chat-template format."""
    return [
        {"role": "system", "content": system_content},
        {"role": "user",   "content": user_content},
    ]

def fetch_response(response):
    """Extract the assistant reply text from a pipeline response."""
    return response[0]["generated_text"][-1]["content"]

## Section 2: Baseline

Without any kind of optimization, we observe the model's response first.

In [ ]:
system_prompt = "You are a helpful assistant."
TASK = "Explain what a neural network is."
response = model_pipeline(make_message(system_prompt, TASK))
baseline_response = fetch_response(response)
print(baseline_response)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A neural network is a type of machine learning model inspired by the structure and function of the human brain. It's a complex system of interconnected nodes or "neurons" that process and transmit information.

**How it works:**

1. **Input layer**: The neural network receives input data, which can be images, text, audio, or any other type of data.
2. **Hidden layers**: The input data is then passed through multiple layers of interconnected nodes (neurons) that apply non-linear transformations to the data. These transformations allow the network to learn complex patterns and relationships in the data.
3. **Output layer**: The final layer of the network produces the output, which can be a prediction, classification, or regression value.

**Key characteristics:**

1. **Artificial neurons**: Each node in the network is called an artificial neuron, which receives one or more inputs, applies a non-linear transformation, and produces an output.
2. **Connections**: Artificial neurons are conn

## Section 3: Role Playing

The system prompt sets the tone and behaviour of the model. By instructing the model to take different personas, it can be prompted for different kinds of responses, as required by the user.

In [ ]:
personas = {
    # Persona A — aimed at absolute beginners, uses analogies
    "Primary School Teacher": (
        "You are a friendly primary school teacher explaining complex topics "
        "using simple everyday analogies. Use short sentences, avoid jargon, "
        "and include a fun analogy."
    ),

    # Persona B — aimed at university students, technical but accessible
    "University Professor": (
        "You are a university professor teaching an introductory machine learning "
        "course. Use precise terminology but explain each term when first introduced. "
        "Structure your answer with a brief overview, core concepts, and a summary."
    ),

    # Persona C — aimed at engineers, code-first, implementation details
    "Senior ML Engineer": (
        "You are a senior ML engineer at a tech company. Give a concise, "
        "technically accurate explanation. Include a tiny Python/pseudocode snippet "
        "to ground the concept. Assume the reader knows Python and basic maths."
    ),
}

persona_responses = {}

for persona_name, system_prompt in personas.items():
    print("\n" + "=" * 60)
    print(f"PERSONA: {persona_name}")
    print("=" * 60)

    response = model_pipeline(make_message(system_prompt, TASK))
    persona_response = fetch_response(response)
    persona_responses[persona_name] = persona_response
    print(persona_response)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PERSONA: Primary School Teacher


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Imagine you have a big box of LEGOs, and inside that box, you have lots of different colored blocks. Each block can be connected to other blocks in different ways.

A neural network is like a super-smart LEGO builder. It takes in lots of different blocks (called "inputs") and tries to figure out what they mean together. It's like trying to build a picture with the blocks.

The builder (the neural network) looks at each block and says, "Oh, this one is a red block, and this one is a blue block. I think I know what they mean together!" And then it connects the blocks in a special way to make a picture.

But here's the really cool thing about the LEGO builder: it can learn and get better at building pictures over time. If you show it lots of different pictures, it can start to recognize patterns and figure out what makes a good picture. And that's kind of like how our brains work, too!

PERSONA: University Professor


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Overview**

A neural network is a fundamental concept in machine learning, inspired by the structure and function of the human brain. It's a computational model designed to recognize patterns, make predictions, and learn from data. Neural networks are composed of interconnected nodes or "neurons" that process and transmit information, allowing them to learn and improve their performance over time.

**Core Concepts**

1. **Artificial Neurons (Nodes)**: The basic building blocks of a neural network, artificial neurons receive one or more inputs, perform a computation on those inputs, and produce an output. Each node applies a non-linear transformation to the inputs, allowing the network to learn complex relationships between variables.
2. **Connections (Edges)**: Artificial neurons are connected to each other through edges, which represent the flow of information between nodes. The strength of these connections can be adjusted during training to optimize the network's performance.
3. *

In [ ]:
baseline_word_count = len(baseline_response.split())
print("Baseline                    ->", baseline_word_count, "words")

for name, resp in persona_responses.items():
    word_count = len(resp.split())
    print(name, "->", word_count, "words")

Baseline                    -> 341 words
Primary School Teacher -> 167 words
University Professor -> 392 words
Senior ML Engineer -> 326 words


## Section 4: Prompt Chaining

Prompt chaining breaks a complex task into a sequence of simpler sub-tasks.

In [ ]:
print("="*20, "\n", "CHAIN STEP 1/3", "\n", "=" * 20)

system_prompt = "You are a helpful assistant."
response = model_pipeline(make_message(system_prompt, TASK))
response1 = fetch_response(response)
print(response1)

print("="*20, "\n", "CHAIN STEP 2/3", "\n", "=" * 20)

system_prompt = "You are an expert technical editor specialising in educational content."
critique_prompt = """Read the following explanation and provide a structured critique.
Identify:
  1. Clarity issues (jargon not explained, confusing sentences)
  2. Missing concepts (what important ideas are absent?)
  3. Audience fit (who is this written for? Is that appropriate?)

<<<DRAFT>>>""" + response1 + """<<<END DRAFT>>>

Output your critique as a numbered list. Be specific and actionable."""

response = model_pipeline(make_message(system_prompt, critique_prompt))
response2 = fetch_response(response)
print(response2)

print("="*20, "\n", "CHAIN STEP 3/3", "\n", "=" * 20)

system_prompt = "You are a skilled technical writer."
refine_prompt = f"""\
You previously wrote a draft explanation and received a critique.
Now write an IMPROVED version that addresses all critique points.

Original task: {TASK}

<<<DRAFT>>>""" + response1 + """<<<END DRAFT>>>

<<<CRITIQUE>>>""" + response2 + """<<<END CRITIQUE>>>

Write the improved explanation. It should be clear, well-structured,
and suitable for a general technical audience."""

response = model_pipeline(make_message(system_prompt, refine_prompt))
response3 = fetch_response(response)
print(response3)

print("="*20, "\n", "Chain complete", "\n", "=" * 20)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 CHAIN STEP 1/3 


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A neural network is a type of machine learning model inspired by the structure and function of the human brain. It's a complex system of interconnected nodes or "neurons" that process and transmit information.

**How it works:**

1. **Input layer**: The neural network receives input data, which can be images, text, audio, or any other type of data.
2. **Hidden layers**: The input data is then passed through multiple layers of "hidden" nodes, which apply non-linear transformations to the data. These transformations allow the network to learn complex patterns and relationships in the data.
3. **Output layer**: The final layer of nodes produces the output of the neural network, which can be a prediction, classification, or any other type of output.

**Key characteristics:**

1. **Artificial neurons**: Each node in the network is called an artificial neuron, which receives one or more inputs, applies a non-linear transformation, and produces an output.
2. **Connections**: Artificial neuron

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here's a structured critique of the provided explanation:

**Clarity Issues:**

1. **Lack of explanation for "non-linear transformations"**: The text assumes that readers are familiar with the concept of non-linear transformations, but it doesn't provide a clear explanation. Consider adding a brief definition or example to help readers understand this concept.
2. **Vague description of "artificial neurons"**: The text mentions that artificial neurons receive inputs, apply transformations, and produce outputs, but it doesn't provide a clear definition of what these components are or how they work. Consider adding a simple diagram or explanation to help readers understand the concept.
3. **Confusing sentence structure**: Some sentences are long and convoluted, making it difficult to follow the text. Consider breaking up long sentences into shorter, simpler ones to improve clarity.
4. **Lack of explanation for "weighted edges"**: The text mentions that artificial neurons are connected thr

In [ ]:
print("Prompt chaining summary")
print("Draft (Step 1) :", len(response1.split()))
print("Critique (Step 2) :", len(response2.split()))
print("Refined (Step 3) :", len(response3.split()))

Prompt chaining summary
Draft (Step 1) : 337
Critique (Step 2) : 392
Refined (Step 3) : 396


## Section 5: Automatic Prompt Engineer

In [ ]:
N_CANDIDATES = 5

system_prompt = "You are an expert prompt engineer."
meta_prompt = "Your job is to generate " + str(N_CANDIDATES) + """ different system-prompt instructions for an AI assistant that must answer the following user task:

TASK: """ + TASK + """

Each instruction should take a different approach (e.g., different audience, different output format, different explanation strategy).

Output only a numbered list (1. ... 2. ... etc.) with one instruction per line.
Each instruction should be 1-2 sentences. No extra commentary."""

print("STEP 1 - Generating", N_CANDIDATES, " Candidate Instructions")

response = model_pipeline(make_message(system_prompt, meta_prompt))
proposals = fetch_response(response)
print(proposals)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STEP 1 - Generating 5  Candidate Instructions
1. For a beginner's audience, explain the concept of a neural network in simple terms, using an analogy and avoiding technical jargon, and provide a brief overview of its applications.
2. Provide a step-by-step explanation of how a neural network is trained, including the process of data preparation, model selection, and optimization, in a formal and technical tone.
3. Create a visual representation of a neural network, labeling its components and explaining their functions, and provide a concise summary of its architecture and capabilities.
4. Write a narrative story about a character who discovers a neural network and its potential to solve complex problems, using a fictional scenario to illustrate the concept and its applications.
5. Offer a comparison between a neural network and other machine learning models, highlighting its strengths and weaknesses, and providing examples of its use in real-world applications.


In [ ]:
import re

# re.findall returns a list of all regex matches in the string
# Pattern: optional whitespace, digit(s), dot, space, then capture the rest of the line
candidates = re.findall(r"^\s*\d+\.\s*(.+)$", proposals, re.MULTILINE)
candidates = candidates[:N_CANDIDATES]

print("Parsed candidate instructions:")
for i, c in enumerate(candidates, 1):
  print(i, ")", c)

Parsed candidate instructions:
1 ) For a beginner's audience, explain the concept of a neural network in simple terms, using an analogy and avoiding technical jargon, and provide a brief overview of its applications.
2 ) Provide a step-by-step explanation of how a neural network is trained, including the process of data preparation, model selection, and optimization, in a formal and technical tone.
3 ) Create a visual representation of a neural network, labeling its components and explaining their functions, and provide a concise summary of its architecture and capabilities.
4 ) Write a narrative story about a character who discovers a neural network and its potential to solve complex problems, using a fictional scenario to illustrate the concept and its applications.
5 ) Offer a comparison between a neural network and other machine learning models, highlighting its strengths and weaknesses, and providing examples of its use in real-world applications.


In [ ]:
print("STEP 2 - Scoring candidates")

# ============ HELPER FUNCTION ============
def score_response(response_text, original_task):
  judge_prompt = "Rate the following response to the task: " + TASK + """
  <<<BEGIN RESPONSE>>>
  """ + response_text[:500] + """
  <<<END RESPONSE>>>

  Score it on these criteria (each 0-2 points):
    1. Accuracy — factually correct?
    2. Clarity — easy to understand?
    3. Completeness — covers the key ideas?
    4. Conciseness — no unnecessary filler?
    5. Structure — well-organised?

Output ONLY a single integer from 1 to 10. Nothing else."""

  response = model_pipeline(make_message(system_prompt, judge_prompt))
  judge_response = fetch_response(response)

  # Parse the score
  match = re.search(r"\d+", judge_response)
  if match:
    raw = int(match.group())
    return float(max(1, min(10, raw)))
  else:
    return 5.0


# ========================================


scored_candidates = []

for idx, instruction in enumerate(candidates):
  print("Candidate", idx + 1, "/", len(candidates))
  print("Instruction:", instruction)

  response = model_pipeline(make_message(instruction, TASK))
  score_response_string = fetch_response(response)

  # Evaluation
  score = score_response(score_response_string, TASK)

  print("Score:", score)
  scored_candidates.append((score, instruction, score_response_string))

scored_candidates.sort(key=lambda x: x[0], reverse=True)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STEP 2 - Scoring candidates
Candidate 1 / 5
Instruction: For a beginner's audience, explain the concept of a neural network in simple terms, using an analogy and avoiding technical jargon, and provide a brief overview of its applications.


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: 7.0
Candidate 2 / 5
Instruction: Provide a step-by-step explanation of how a neural network is trained, including the process of data preparation, model selection, and optimization, in a formal and technical tone.


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Score: 8.0
Candidate 4 / 5
Instruction: Write a narrative story about a character who discovers a neural network and its potential to solve complex problems, using a fictional scenario to illustrate the concept and its applications.


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: 8.0
Candidate 5 / 5
Instruction: Offer a comparison between a neural network and other machine learning models, highlighting its strengths and weaknesses, and providing examples of its use in real-world applications.


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: 8.0


In [ ]:
print("STEP 3 - Results")

best_score, best_instruction, best_response = scored_candidates[0]
for rank, (score, instr, _) in enumerate(scored_candidates, 1):
  print("Rank", rank, "(score=", score, ")", instr[:80], "...")

print("Best Instruction:", best_instruction)
print("Score:", best_score, "/ 10")
print("Best response:")
print("=" * 20)
print(best_response)
print("=" * 20)

STEP 3 - Results
Rank 1 (score= 8.0 ) Provide a step-by-step explanation of how a neural network is trained, including ...
Rank 2 (score= 8.0 ) Create a visual representation of a neural network, labeling its components and  ...
Rank 3 (score= 8.0 ) Write a narrative story about a character who discovers a neural network and its ...
Rank 4 (score= 8.0 ) Offer a comparison between a neural network and other machine learning models, h ...
Rank 5 (score= 7.0 ) For a beginner's audience, explain the concept of a neural network in simple ter ...
Best Instruction: Provide a step-by-step explanation of how a neural network is trained, including the process of data preparation, model selection, and optimization, in a formal and technical tone.
Score: 8.0 / 10
Best response:
**Introduction to Neural Networks**

A neural network is a machine learning model inspired by the structure and function of the human brain. It consists of interconnected nodes or "neurons" that process and transmit informa

## Section 6: Gradient-based textural optimization

ProTeGi defines "textual gradients" which are natural-language feedback that acts like gradient descent.

In [ ]:
T = 3          # Number of optimization steps
BEAM_WIDTH = 3 # Number of candidates per step

initial_prompt = "Answer the following question."

quality_criteria = """A good response should:
  - Be understandable to a non-expert
  - Use at least one concrete analogy
  - Be structured (introduction, body, summary)
  - Avoid unexplained technical jargon"""

### Helper functions for ProTeGi

In [ ]:
def compute_textual_gradient(prompt, response, task, criteria):
  system_prompt = "You are an expert in prompt engineering."
  gradient_prompt = """You are a prompt-quality analyser.

The following instruction was used to guide a language model:
INSTRUCTION: """ + prompt + """
The model responded to the task ===""" + task + """=== with:
<<<BEGIN RESPONSE>>>
""" + response[:400] + """
<<<END RESPONSE>>> (truncated)

Quality criteria the response should meet:
""" + criteria + """

Identify exactly which criteria are NOT met and explain WHY the current instruction fails to elicit them. Be specific and actionable. Output 2-4 bullet points only."""
  response = model_pipeline(make_message(system_prompt, gradient_prompt))
  return fetch_response(response)

def edit_prompt_with_gradient(prompt, gradient, beam_width):
  system_prompt = "You are an expert prompt engineer."
  edit_prompt_text = """You are a prompt engineer. Improve the following instruction based on feedback.

CURRENT INSTRUCTION: ===""" + prompt + """===

FEEDBACK (what's wrong and why):
""" + gradient + """

Generate """ + str(beam_width) + """ improved versions of the instruction. Each version should directly address the feedback. Output ONLY a numbered list (1. ... 2. ... etc.), one instruction per line. Keep each instruction under 3 sentences."""
  response = model_pipeline(make_message(system_prompt, edit_prompt_text))
  edit_response = fetch_response(response)

  edits = re.findall(r"^\s*\d+\.\s*(.+)$", edit_response, re.MULTILINE)

  if not edits: # In case model failed
    edits = [prompt + " Provide a clear, well-structured explanation."]

  return edits[:beam_width]

In [ ]:
current_prompt = initial_prompt
history = []

for iteration in range(T):
  print("\n" + "═" * 60)
  print(f"ITERATION {iteration + 1} / {T}")
  print("═" * 60)
  print(f"Current prompt: '{current_prompt}'")

  print("\n Generating response...")
  response = model_pipeline(make_message(current_prompt, TASK))
  current_response = fetch_response(response)

  current_score = score_response(current_response, TASK)
  print("Score:", current_score, "/ 10")
  history.append((iteration, current_prompt, current_score))

  print("Computing gradient...")
  gradient = compute_textual_gradient(current_prompt, current_response, TASK, quality_criteria)
  print(gradient)

  print("Generating", BEAM_WIDTH, "candidate edits...")
  candidates = edit_prompt_with_gradient(current_prompt, gradient, BEAM_WIDTH)

  print("Scoring candidates...")
  best_candidate_score = -1
  best_candidate_prompt = current_prompt

  for j, candidate in enumerate(candidates):
    response = model_pipeline(make_message(candidate, TASK))
    cand_response = fetch_response(response)

    cand_score = score_response(cand_response, TASK)
    print("    Candidate", j+1, ": score=", cand_score)

    if cand_score > best_candidate_score:
      best_candidate_score = cand_score
      best_candidate_prompt = candidate

  if best_candidate_score > current_score:
    print("Accepted update:", current_score, "->", best_candidate_score)
    current_prompt = best_candidate_prompt
  else:
    print("No improvement found (keeping current prompt)")

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



════════════════════════════════════════════════════════════
ITERATION 1 / 3
════════════════════════════════════════════════════════════
Current prompt: 'Answer the following question.'

 Generating response...


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: 8.0 / 10
Computing gradient...


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    Candidate 1 : score= 8.0


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Candidate 2 : score= 8.0


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Candidate 3 : score= 7.0
No improvement found (keeping current prompt)


In [ ]:
print("="*20, "\n", "Final response with optimized prompt", "\n", "="*20)
response = model_pipeline(make_message(current_prompt, TASK))
best_response = fetch_response(response)

print(best_response)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Final response with optimized prompt 
A neural network is a computer system inspired by the structure and function of the human brain. It's a type of machine learning model that's composed of interconnected nodes or "neurons," which process and transmit information.

In a neural network, the nodes are arranged in layers, with each layer processing and transforming the input data in a specific way. The nodes are connected by edges, which represent the strength and direction of the connections between them. This allows the network to learn and represent complex patterns in the data.

Here's a simplified overview of how a neural network works:

1. **Input layer**: The network receives input data, which is fed into the input layer.
2. **Hidden layers**: The input data is then processed by one or more hidden layers, where the nodes apply non-linear transformations to the data.
3. **Output layer**: The final output is generated by the output layer, which takes the transformed data from the 

## Section 7 [**Try yourself**] - DSPy - _programming_ prompts instead of writing them

[DSPy](https://dspy.ai) turns that idea into a reusable framework. Instead of writing
prompt *strings*, you:

1. **Declare a `Signature`** — *what* goes in and *what* comes out (e.g. `question -> explanation`).
2. **Compose `Module`s** — building blocks like `dspy.Predict` or `dspy.ChainOfThought`
   that turn a Signature into a callable program.
3. **Compile with an `Optimizer`** (a.k.a. *teleprompter*) — it automatically searches for
   the best prompt *text* and few-shot *examples* against a metric you define.

In short: you describe the task, and DSPy writes (and tunes) the prompt for you. APE and
ProTeGi were essentially special cases of this idea — DSPy gives you the general-purpose tool.

In [ ]:
!pip install dspy

import dspy
print("DSPy version:", dspy.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.4 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.2
    Uninstalling typeguard-4.5.2:
      Successfully uninstalled typeguard-4.5.2


05:34:38 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
05:34:39 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


DSPy version: 3.2.1


### Step 1 - Create DSPy.LM object from our predefined pipeline

In [ ]:
from litellm import ModelResponse
from litellm.types.utils import Choices, Message, Usage


class LocalPipelineLM(dspy.LM):
    """Adapter that lets DSPy drive our local HuggingFace `model_pipeline`."""

    def __init__(self, pipeline, **kwargs):
        super().__init__(model="unsloth/Llama-3.2-3B-Instruct", model_type="chat", **kwargs)
        self.pipeline = pipeline

    def forward(self, prompt=None, messages=None, **kwargs):
        # DSPy hands us chat-style messages; reuse our existing helper to read the reply.
        messages = messages or [{"role": "user", "content": prompt}]
        text = fetch_response(self.pipeline(messages))
        # Return an OpenAI-compatible response object (what DSPy expects).
        return ModelResponse(
            choices=[Choices(message=Message(role="assistant", content=text))],
            model=self.model,
            usage=Usage(prompt_tokens=0, completion_tokens=0, total_tokens=0),
        )


# Register our local model as the default LM for all DSPy modules.
dspy.configure(lm=LocalPipelineLM(model_pipeline))
print("DSPy is now using our local Llama pipeline.")

DSPy is now using our local Llama pipeline.


### Step 2 - Declare the task as a `Signature`

A `Signature` is a typed `input -> output` contract.

In [ ]:
class ExplainConcept(dspy.Signature):
    """Explain a technical concept clearly for a non-expert: use a concrete analogy,
    a simple structure (intro, body, summary), and avoid unexplained jargon."""

    question: str = dspy.InputField(desc="the concept or question to explain")
    explanation: str = dspy.OutputField(desc="a clear, well-structured explanation")

### Step 3 - Build a module and run it zero-shot

`dspy.Predict` turns the Signature into a callable program.

In [ ]:
program = dspy.Predict(ExplainConcept)

zeroshot = program(question=TASK)
print("=== DSPy zero-shot output ===")
print(zeroshot.explanation)

zeroshot_score = score_response(zeroshot.explanation, TASK)
print("\nZero-shot DSPy score:", zeroshot_score, "/ 10")

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== DSPy zero-shot output ===
A neural network is a computer system inspired by the structure and function of the human brain. It's composed of layers of interconnected nodes or "neurons," which process and transmit information.

Imagine a network of people passing messages to each other. Each person (node) receives a message, performs a simple operation on it, and then passes the result to the next person in the chain. This process continues until the message reaches its final destination.

In a neural network, these nodes are arranged in layers, with each layer processing different aspects of the input data. The nodes in one layer send their outputs to the nodes in the next layer, allowing the network to learn and represent complex patterns in the data.

The key characteristics of neural networks are:

*   **Artificial neurons**: The nodes that process and transmit information.
*   **Connections**: The links between nodes that allow them to exchange information.
*   **Layers**: The h

### Step 4 - Let an optimizer tune the prompt

To optimize, DSPy needs a **metric** and a few **training examples**. We reuse our judge as
the metric. `BootstrapFewShot` then runs the program on the examples, keeps the attempts
that *pass* the metric, and bakes them into the prompt as few-shot demonstrations —
automatically.

In [ ]:
def quality_metric(example, prediction, trace=None):
    """Pass if the judge rates the explanation at least 7/10."""
    return score_response(prediction.explanation, example.question) >= 7.0


# A handful of example questions (no hand-written prompts needed).
trainset = [
    dspy.Example(question="Explain what a neural network is.").with_inputs("question"),
    dspy.Example(question="Explain what gradient descent is.").with_inputs("question"),
    dspy.Example(question="Explain what overfitting is.").with_inputs("question"),
    dspy.Example(question="Explain what a loss function is.").with_inputs("question"),
]

from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=quality_metric,
    max_bootstrapped_demos=2,
    max_labeled_demos=2,
)
optimized_program = optimizer.compile(program, trainset=trainset)

print("Optimization complete. Demos auto-selected:", len(optimized_program.demos))


  0%|          | 0/4 [00:00<?, ?it/s]Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)

 25%|██▌       | 1/4 [00:12<00:37, 12.62s/it]Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for mo

Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Optimization complete. Demos auto-selected: 2


### Step 5 - Compare, and inspect the prompt built by DSPy

In [ ]:
optimized = optimized_program(question=TASK)

print("=== DSPy OPTIMIZED output ===")
print(optimized.explanation)

optimized_score = score_response(optimized.explanation, TASK)
print("\nOptimized DSPy score:", optimized_score, "/ 10")
print("Zero-shot was:       ", zeroshot_score, "/ 10")

# Peek at the exact prompt DSPy sent — note the few-shot examples it inserted for us.
print("\n--- The prompt DSPy generated (most recent call) ---")
dspy.inspect_history(n=1)

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== DSPy OPTIMIZED output ===
A neural network is a computer system inspired by the structure and function of the human brain. It's composed of layers of interconnected nodes or "neurons," which process and transmit information.

Imagine a network of interconnected mailboxes, where each mailbox represents a node. When you send a letter (or data) to a mailbox, it's processed by the node, and then forwarded to other mailboxes, creating a flow of information. This flow of information is similar to how neurons communicate with each other in the brain.

In a neural network, each node applies a set of rules (or weights) to the incoming data, and then passes the result to other nodes. This process is repeated, allowing the network to learn and make decisions based on the data it receives.

The key characteristics of neural networks are:

*   **Artificial neurons**: The nodes that process and transmit information.
*   **Connections**: The links between nodes that allow them to communicate.
*  

- We did not write a prompt string. We declared a **Signature**, picked a **Module**
  (`dspy.Predict`), and let an **Optimizer** assemble the prompt — including few-shot
  examples it selected automatically using our metric.
- `dspy.inspect_history` shows the real prompt: DSPy rewrote our task into a structured
  format and injected worked examples — exactly what we did by hand in APE and ProTeGi,
  but generated for us.

## Section 8 - Comparison of all the approaches

In [ ]:
# ── Final comparison: run all techniques and score them ───────────────────────

techniques = {
    "1. Baseline (naive)" : "You are a helpful assistant.",
    "2. Role-playing"     : personas["University Professor"],
    "3. Chaining" : "You are a skilled technical writer."  # refined output uses this
                            " Write explanations that are clear, structured, and accessible.",
    "4. APE"       : best_instruction,
    "5. ProTeGi (final)"  : current_prompt,
}

print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

results = {}
for name, prompt in techniques.items():
    response = model_pipeline(make_message(prompt, TASK))
    resp = fetch_response(response)
    score = score_response(resp, TASK)
    results[name] = (score, prompt)

# Sort by score
sorted_results = sorted(results.items(), key=lambda x: x[1][0], reverse=True)

print(f"\n{'Technique':<30} {'Score':>6}  Prompt (truncated)")
print("-" * 80)
for name, (score, prompt) in sorted_results:
    bar    = "★" * int(round(score))
    prompt_short = prompt[:50].replace("\n", " ")
    print(f"  {name:<28} {score:>5.1f}  {bar}")
    print(f"  {'':28}        Prompt: '{prompt_short}...'")

winner_name, (winner_score, winner_prompt) = sorted_results[0]
print(f"\n Overall winner: {winner_name} (score: {winner_score}/10)")

Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINAL COMPARISON


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Technique                       Score  Prompt (truncated)
--------------------------------------------------------------------------------
  1. Baseline (naive)            8.0  ★★★★★★★★
                                      Prompt: 'You are a helpful assistant....'
  2. Role-playing                8.0  ★★★★★★★★
                                      Prompt: 'You are a university professor teaching an introdu...'
  3. Chaining                    8.0  ★★★★★★★★
                                      Prompt: 'You are a skilled technical writer. Write explanat...'
  4. APE                         8.0  ★★★★★★★★
                                      Prompt: 'Provide a step-by-step explanation of how a neural...'
  5. ProTeGi (final)             8.0  ★★★★★★★★
                                      Prompt: 'Answer the following question....'

 Overall winner: 1. Baseline (naive) (score: 8.0/10)


## Summary & Further Reading

| Technique | What it does | Effort | When to use |
|-----------|-------------|--------|-------------|
| **Baseline** | Plain prompt | Minimal | Prototyping |
| **Role Playing** | Persona in system prompt | Low | Audience-specific responses |
| **Prompt Chaining** | Multi-step pipeline | Medium | Complex, multi-part tasks |
| **APE** | Auto-generate + score many prompts | High (LLM calls) | When you have evaluation data |
| **ProTeGi** | Iterative feedback-based editing | High (LLM calls) | Fine-tuning an existing good prompt |
| **DSPy** | Declare the task; the framework writes & tunes the prompt | Medium–High | Reusable, metric-driven optimization at scale |

### Key Papers
- **APE**: Zhou et al. (2023) — *Large Language Models Are Human-Level Prompt Engineers* — ICLR 2023. [arxiv.org/abs/2211.01910](https://arxiv.org/abs/2211.01910)
- **ProTeGi**: Pryzant et al. (2023) — *Automatic Prompt Optimization with Gradient Descent and Beam Search* — EMNLP 2023. [arxiv.org/abs/2305.03495](https://arxiv.org/abs/2305.03495)
- **DSPy**: Khattab et al. (2023) — *DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines* — ICLR 2024. [arxiv.org/abs/2310.03714](https://arxiv.org/abs/2310.03714)
- **Chain-of-Thought**: Wei et al. (2022) — *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* — NeurIPS 2022.

### Useful Resources
- [Anthropic Prompt Engineering Guide](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)
- [Unsloth Documentation](https://github.com/unslothai/unsloth)
- [LangChain — for production prompt chaining](https://python.langchain.com)
- [DSPy Documentation](https://dspy.ai)
